In [ ]:
!pip install laspy rasterio

In [ ]:
import laspy
import rasterio
import numpy as np
import os

def normalize_vegetation_heights(laz_file: str, dtm_tif: str, output_laz: str) -> None:
    """
    Normalizes vegetation point heights using a given DTM.

    Caution: As comments in Line 39-46 shows, for difference LiDAR datasets, the vegetation
             points should be correctly specified.
    
    Parameters
    ----------
    laz_file : str
        Path to the input .laz file (LiDAR point cloud).
    dtm_tif : str
        Path to the DTM (GeoTIFF format).
    output_laz : str
        Path to save the normalized .laz file.

    Returns
    -------
    None
        Saves a new LiDAR file with normalized heights.
    """
    try:
        # Load DTM
        with rasterio.open(dtm_tif) as dtm:
            dtm_data = dtm.read(1)  # Read elevation values
            dtm_transform = dtm.transform  # Get spatial transform
            dtm_bounds = dtm.bounds  # Get extent
            dtm_res = dtm.res  # Resolution
            print(dtm_res)

        # Load LiDAR data
        with laspy.open(laz_file) as las_file:
            las = las_file.read()

        # Here, normally in standard LAZ/LAS files, vegetation points were classified as [3, 4, 5].
        # However, Dutch AHN point clouds donot have vegetation points classified, they are unclassified as [1]
        
        # Extract vegetation points (classification: 3, 4, 5)
        # vegetation_mask = np.isin(las.classification, [3, 4, 5])
        
        # For Dutch AHN case, should be used as below:
        vegetation_mask = np.isin(las.classification, [2])

        # Convert ScaledArrayView to NumPy arrays
        x_veg = np.array(las.x[vegetation_mask])
        y_veg = np.array(las.y[vegetation_mask])
        z_veg = np.array(las.z[vegetation_mask])

        # Convert LiDAR coordinates to raster grid indices
        row_indices = ((dtm_bounds.top - y_veg) / dtm_res[1]).astype(int)
        col_indices = ((x_veg - dtm_bounds.left) / dtm_res[0]).astype(int)

        # Ensure indices are within DTM bounds
        valid_mask = (row_indices >= 0) & (row_indices < dtm_data.shape[0]) & \
                     (col_indices >= 0) & (col_indices < dtm_data.shape[1])

        # Compute normalized heights
        normalized_heights = z_veg[valid_mask] - dtm_data[row_indices[valid_mask], col_indices[valid_mask]]

        # Create new LAS file with normalized heights
        header = laspy.LasHeader(point_format=las.header.point_format, version=las.header.version)
        header.scales = las.header.scales
        header.offsets = las.header.offsets
        
        normalized_las = laspy.LasData(header)
        normalized_las.points = las.points[vegetation_mask][valid_mask]  # Assign filtered points
        normalized_las.z = normalized_heights  # Replace z values with normalized height

        # Save output LAS file
        with laspy.open(output_laz, mode="w", header=header) as las_writer:
            las_writer.write_points(normalized_las.points)

        print(f"Normalized LiDAR file saved: {output_laz}")

    except Exception as e:
        raise Exception(f"Error in normalizing vegetation heights: {e}")


In [ ]:
laz_file = "../../Datasets/24HN1_25.laz"  # Input LiDAR file
dtm_tif = "../../Datasets/24HN1_25_1m_IDW.tif"  # Input DTM file (GeoTIFF)
output_laz = "../../Datasets/normalized_24HN1_25_1m_IDW.laz"  # Output file

# Normalize vegetation heights
normalize_vegetation_heights(laz_file, dtm_tif, output_laz)